In [0]:

SHOW TABLES IN pondolens.dev_bronze;

database,tableName,isTemporary
dev_bronze,agency_actuals,false
dev_bronze,ched_graduates,false
dev_bronze,gdrive_ingestion_log,false
dev_bronze,grdp,false
dev_bronze,grdp_per_capita,false
dev_bronze,population,false
dev_bronze,poverty_incidence,false
dev_bronze,regional_expenditure_gaa,false
dev_bronze,regional_expenditure_obligations,false
dev_bronze,sector_expenditure,false


### 1. Check schema

In [0]:
DESCRIBE pondolens.dev_bronze.grdp;

col_name,data_type,comment
region_name,string,null
year,string,null
grdp_current_prices_thousand_php,string,null
grdp_constant_2018_prices_thousand_php,string,null
geographic_level,string,null
source_file,string,null
source_release,string,null
_rescued_data,string,null
_drive_file_id,string,null
_source_file,string,null


### 2. Compare overall rows versus 2021–2025

In [0]:
SELECT
    COUNT(*) AS overall_rows,
    COUNT_IF(TRY_CAST(year AS INT) BETWEEN 2021 AND 2025)
        AS rows_2021_2025,
    COUNT_IF(
        TRY_CAST(year AS INT) < 2021
        OR TRY_CAST(year AS INT) > 2025
    ) AS rows_outside_scope,
    COUNT_IF(TRY_CAST(year AS INT) IS NULL)
        AS missing_or_unparseable_years
FROM pondolens.dev_bronze.grdp;

overall_rows,rows_2021_2025,rows_outside_scope,missing_or_unparseable_years
95,95,0,0


### 3. Check rows and geography count per year

In [0]:
SELECT
    year,
    geographic_level,
    COUNT(*) AS row_count,
    COUNT(DISTINCT region_name) AS geography_count
FROM pondolens.dev_bronze.grdp
GROUP BY year, geographic_level
ORDER BY year, geographic_level;

year,geographic_level,row_count,geography_count
2021,national,1,1
2021,region,18,18
2022,national,1,1
2022,region,18,18
2023,national,1,1
2023,region,18,18
2024,national,1,1
2024,region,18,18
2025,national,1,1
2025,region,18,18


### 4. Check actual geography labels, including NIR and national total

In [0]:
SELECT
    region_name,
    geographic_level,
    COUNT(*) AS row_count,
    COUNT(DISTINCT year) AS years_present
FROM pondolens.dev_bronze.grdp
GROUP BY region_name, geographic_level
ORDER BY geographic_level, region_name;

region_name,geographic_level,row_count,years_present
Philippines,national,5,5
Bangsamoro Autonomous Region in Muslim Mindanao,region,5,5
Bicol Region,region,5,5
CALABARZON,region,5,5
Cagayan Valley,region,5,5
Caraga,region,5,5
Central Luzon,region,5,5
Central Visayas,region,5,5
Cordillera Administrative Region,region,5,5
Davao Region,region,5,5


### 5. Check source versions

In [0]:
SELECT
    source_file,
    source_release,
    _dataset_hash,
    _run_id,
    COUNT(*) AS row_count
FROM pondolens.dev_bronze.grdp
GROUP BY
    source_file,
    source_release,
    _dataset_hash,
    _run_id
ORDER BY source_release, _run_id;

source_file,source_release,_dataset_hash,_run_id,row_count
2018-2025 GRDP_Reg_(with NIR as of April 2026)(1).xlsx,2026-04,06a818122267eb7adb969eb4dbccea758fb4e2c788b76ab90cbe7e68bb541c7a,46a8d68d06984bcdb85d0c4bbadb91d3,95


## Final findings

The completed profiling checks cover 95 records for 2021–2025: 18 regions and one national total per year, including Negros Island Region.

- No missing required values, invalid years or amounts, zero amounts, duplicate keys, missing geography-year combinations, label issues, rescued rows, or missing checked lineage fields were reported.
- Regional sums match national totals for both GRDP measures in all five years.
- All 152 computed growth values match the original workbook at two decimal places: 19 geographies × 4 annual comparisons × 2 measures.
- Western Visayas 2022 exceeded the proposed 20% review threshold: 25.22% current-price growth and 10.23% constant-price growth. Both match the source; retain the record. No quarantine is required for this finding.
- Growth for 2021 is not applicable because 2020 is outside the ingested scope.

Source units are **thousand Philippine pesos**. The source assigns Sulu to Zamboanga Peninsula throughout 2018–2025. Confirm regional boundaries before joining with population or budget data.

These findings summarize the completed checks and source comparison; the detailed temporary-view queries have been removed.


## Proposed Silver columns

**Grain:** one row per geography, year, and source release.

| Column | Proposed type | Purpose |
|---|---|---|
| `region_code` | STRING | Standard geography code from a verified mapping |
| `region_name` | STRING | Standardized geography name |
| `geographic_level` | STRING | `region` or `national` |
| `year` | INT | Reporting year |
| `grdp_current_thousand_php` | DECIMAL(p, s) | GRDP at current prices, in thousand PHP |
| `grdp_constant_2018_thousand_php` | DECIMAL(p, s) | GRDP at constant 2018 prices, in thousand PHP |
| `boundary_version` | STRING | Verified geographic definition used by the source |
| `quality_flags` | ARRAY<STRING> | Recorded quality or review status |
| `source_release` | STRING | Publication version, e.g. `2026-04` |
| `source_file` | STRING | Original source filename |
| `drive_file_id` | STRING | Ingested Google Drive file ID |
| `run_id` | STRING | Bronze ingestion run ID |
| `ingested_at` | TIMESTAMP | Original Bronze ingestion timestamp |

Finalize decimal precision and scale from source values before conversion. Populate region codes and boundary versions only from verified mappings.

Retain `dataset_hash` in Bronze; omit it from Silver. Calculate growth in Gold. Keep national rows, but filter to `geographic_level = 'region'` when summing regional values to avoid double counting. Record Western Visayas 2022 as `SOURCE_VERIFIED_LARGE_CHANGE` in the review log.
